# vLLM Lab Notebook

This notebook has **two parts**:

1. **Part 1: vLLM Toolbox (minimal code)**  
   Small, readable, runnable helpers for:
   - listing models
   - chat completions calls
   - token counting
   - `/metrics` parsing
   - basic latency / throughput measurement
   - per-request **thinking mode** switch via `chat_template_kwargs`

2. **Part 2: Lab Tasks + Hints + Report Templates**  
   Three experiments on **one vLLM service** and **one model**.

---

## Start the only server used in this lab
```bash
vllm serve /home/enine/model/Qwen3-0.6B --port 8000 --api-key token-abc123
```

## Important lab rule
Use **`max_tokens = 4096` in all experiments** (because this is a thinking model).  
If you need short outputs, enforce it through the **prompt**, not by lowering `max_tokens`.


## Part 1. vLLM Toolbox (minimal, readable, reliable)
Suggested order: **config → model list → single request → thinking ON/OFF request → token count → metrics → batch test**


In [36]:
# 1) Basic configuration
import json, time, re, math
from statistics import mean
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    import requests
except ImportError as e:
    raise ImportError("Please install requests first: pip install requests") from e

BASE_URL = "http://localhost:8000"
API_KEY = "token-abc123"
MODEL_PATH = "/home/enine/model/Qwen3-0.6B"  # For documentation only (server is already running)
HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

# Global lab requirement
MAX_TOKENS = 10240

print("BASE_URL =", BASE_URL)
print("MODEL_PATH =", MODEL_PATH)
print("MAX_TOKENS =", MAX_TOKENS)


BASE_URL = http://172.16.78.36:8004
MODEL_PATH = Qwen3-0.6B
MAX_TOKENS = 10240


In [4]:
# 2) Minimal HTTP helpers
import requests

# 1. 定义基础 URL（用 httpbin 测试 API，确保能访问）
BASE_URL = "https://httpbin.org"

# 2. Minimal HTTP helpers
def _url(path, base_url=BASE_URL):
    return base_url.rstrip("/") + "/" + path.lstrip("/")

def get_json(path, *, headers=None, timeout=30):
    r = requests.get(_url(path), headers=headers, timeout=timeout)
    r.raise_for_status()  # 若状态码不是 200-299，抛出异常
    return r.json()

def post_json(path, payload, *, headers=None, timeout=120):
    r = requests.post(_url(path), json=payload, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()


# 3. 测试调用（关键：路径要和 httpbin 的接口匹配！）
if __name__ == "__main__":
    # 测试 GET 请求（httpbin 的 /get 接口）
    try:
        get_data = get_json("get", headers={"User-Agent": "test-client"})
        print("✅ GET 请求成功，返回数据：\n", get_data)
    except Exception as e:
        print("❌ GET 请求失败：", e)

    # 测试 POST 请求（httpbin 的 /post 接口）
    try:
        payload = {"title": "Test", "body": "Hello World", "userId": 1}
        post_data = post_json("post", payload, headers={"Content-Type": "application/json"})
        print("\n✅ POST 请求成功，返回数据：\n", post_data)
    except Exception as e:
        print("❌ POST 请求失败：", e)


✅ GET 请求成功，返回数据：
 {'args': {}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate, br, zstd', 'Host': 'httpbin.org', 'User-Agent': 'test-client', 'X-Amzn-Trace-Id': 'Root=1-69c6485d-586b4069000ff27c45b940c4'}, 'origin': '141.11.146.57', 'url': 'https://httpbin.org/get'}

✅ POST 请求成功，返回数据：
 {'args': {}, 'data': '{"title": "Test", "body": "Hello World", "userId": 1}', 'files': {}, 'form': {}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate, br, zstd', 'Content-Length': '53', 'Content-Type': 'application/json', 'Host': 'httpbin.org', 'User-Agent': 'python-requests/2.32.3', 'X-Amzn-Trace-Id': 'Root=1-69c6485e-7f51691e4028e8266b304532'}, 'json': {'body': 'Hello World', 'title': 'Test', 'userId': 1}, 'origin': '141.11.146.57', 'url': 'https://httpbin.org/post'}


In [3]:
# 3) List models (OpenAI-compatible endpoint)
models_payload = get_json("/v1/models", headers=HEADERS, timeout=20)
print(json.dumps(models_payload, ensure_ascii=False, indent=2)[:1200])

MODEL_ID = models_payload["data"][0]["id"]
print("MODEL_ID =", MODEL_ID)


NameError: name 'HEADERS' is not defined

### Optional curl reference (thinking mode switch)
You can also call the server directly with curl and set thinking mode per request:
```bash
curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "YOUR_MODEL_ID",
    "messages": [{"role": "user", "content": "Your question here"}],
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "max_tokens": 4096,
    "presence_penalty": 1.5,
    "chat_template_kwargs": {"enable_thinking": false}
  }'
```


In [10]:
# 4) Minimal chat/completions call with thinking-mode switch
import requests
import time
import os
from typing import Optional, Dict, Any

# ====== 配置区域 - 根据你的 vLLM 服务修改 ======
API_BASE_URL = "http://172.16.78.36:8004"  # vLLM 服务地址
API_KEY = "token-abc123"  # 与 vLLM 启动时的 --api-key 一致
MODEL_ID = "Qwen3-0.6B"   # 模型 ID（从/v1/models 获取）
MAX_TOKENS = 10240
# ============================================

# 2. 📦 HTTP 请求辅助函数
def _url(path, base_url=API_BASE_URL):
    """构建完整 URL"""
    return base_url.rstrip("/") + "/" + path.lstrip("/")

def get_json(path, *, headers=None, timeout=30, base_url=API_BASE_URL):
    """发送 GET 请求并返回 JSON"""
    r = requests.get(_url(path, base_url=base_url), headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()

def post_json(path, payload, *, headers=None, timeout=120, base_url=API_BASE_URL):
    """发送 POST 请求并返回 JSON"""
    r = requests.post(_url(path, base_url=base_url), json=payload, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()

# 3. 🔧 请求头配置
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# 4. 💬 chat_once 函数 - 支持 thinking mode 切换
def chat_once(
    prompt,
    *,
    model=MODEL_ID,
    temperature=0.2,
    top_p=1.0,
    top_k=None,
    max_tokens=MAX_TOKENS,
    presence_penalty=None,
    enable_thinking=None,   # None = omit; True/False = set chat_template_kwargs.enable_thinking
    timeout=180,
    extra_body=None,
):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
    }
    if top_k is not None:
        payload["top_k"] = top_k
    if presence_penalty is not None:
        payload["presence_penalty"] = presence_penalty
    if enable_thinking is not None:
        payload["chat_template_kwargs"] = {"enable_thinking": bool(enable_thinking)}
    if extra_body:
        payload.update(extra_body)

    t0 = time.perf_counter()
    data = post_json("/v1/chat/completions", payload, headers=HEADERS, timeout=timeout)
    dt = time.perf_counter() - t0

    msg = data["choices"][0]["message"]
    text = msg.get("content", "")
    usage = data.get("usage", {})
    return {"text": text, "usage": usage, "latency_s": dt, "raw": data}

# 5. 🧪 测试运行
if __name__ == "__main__":
    try:
        # 先测试获取模型列表
        models = get_json("/v1/models", headers=HEADERS, timeout=20)
        print("可用模型:", [m["id"] for m in models.get("data", [])])
        
        # 使用简单提示测试
        demo = chat_once("请用一句话介绍你自己。", enable_thinking=False)
        print("\n✅ 调用成功！")
        print(f"延迟：{round(demo['latency_s'], 3)} 秒")
        print(f"Token 使用情况：{demo.get('usage', {})}")
        print(f"回复内容：{demo.get('text', '')[:200]}")
        
    except requests.exceptions.ConnectionError as e:
        print(f"❌ 网络连接失败：{e}")
        print(f"请检查 vLLM 服务是否正在运行：{API_BASE_URL}")
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP 错误：{e}")
        if e.response.status_code == 401:
            print("认证失败：请检查 API_KEY 是否正确")
        elif e.response.status_code == 404:
            print("接口不存在：请检查 API_BASE_URL 是否正确")
        else:
            print(f"HTTP 状态码：{e.response.status_code}")
    except KeyError as e:
        print(f"❌ 响应格式错误：缺少字段 {e}")
    except Exception as e:
        print(f"❌ 未知错误：{type(e).__name__}: {e}")

❌ 网络连接失败：HTTPConnectionPool(host='172.16.78.36', port=8004): Max retries exceeded with url: /v1/models (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000161DD9D1D00>: Failed to establish a new connection: [WinError 10061] 由于目标计算机积极拒绝，无法连接。'))
请检查 vLLM 服务是否正在运行：http://172.16.78.36:8004


In [41]:
# 5) Thinking ON/OFF demo on the same question (format-controlled)
math_demo_prompt = (
    "Solve the problem and output ONLY the final answer in the format '#### <number>'.\n"
    "Question: A bakery made 48 muffins in the morning and 36 in the afternoon. "
    "They sold 59 muffins. How many muffins are left?"
)

out_on = chat_once(math_demo_prompt, temperature=0.0, top_p=1.0, enable_thinking=True, max_tokens=MAX_TOKENS)
out_off = chat_once(math_demo_prompt, temperature=0.0, top_p=1.0, enable_thinking=False, max_tokens=MAX_TOKENS)

print("Thinking ON latency:", round(out_on["latency_s"], 3), "s")
print("Thinking ON output :", out_on["text"])
print("\nThinking OFF latency:", round(out_off["latency_s"], 3), "s")
print("Thinking OFF output :", out_off["text"])


Thinking ON latency: 0.635 s
Thinking ON output : <think>
Okay, let's see. The problem says that a bakery made 48 muffins in the morning and 36 in the afternoon. Then they sold 59 muffins. I need to find out how many muffins are left. Hmm, so first, I should probably figure out the total number of muffins made and then subtract the ones sold. 

Let me start by adding the muffins made in the morning and afternoon. So 48 plus 36. Let me do that math. 48 plus 30 is 78, and then plus 6 more is 84. So total made is 84 muffins. 

Now, they sold 59 muffins. So the remaining ones would be 84 minus 59. Let me calculate that. 84 minus 50 is 34, and then minus 9 more is 25. So 25 muffins are left. 

Wait, let me check again to make sure I didn't make a mistake. Total made: 48 + 36. Yes, 48 + 36. Let me add them another way. 40 + 30 is 70, and 8 + 6 is 14, so total 84. Correct. Then 84 minus 59. 84 - 50 is 34, minus 9 is 25. Yep, that seems right. 

So the answer should be 25. I think that's it. N

In [42]:
# 6) Token counting (two practical ways)
# 6.1 Easiest: read usage from the response
x = chat_once("Write one short slogan about fast inference. One sentence only.", enable_thinking=False)
print("usage from response =", x["usage"])

# 6.2 Optional: /tokenize endpoint for prompt token estimation
def tokenize_count(text, *, model=MODEL_ID, add_special_tokens=True):
    payload = {
        "model": model,
        "prompt": text,
        "add_special_tokens": add_special_tokens,
    }
    r = requests.post(_url("/tokenize"), json=payload, timeout=30)  # usually no auth required
    r.raise_for_status()
    data = r.json()
    token_ids = data.get("token_ids") or data.get("tokens") or []
    return {"count": len(token_ids), "raw": data}

tok = tokenize_count("vLLM can serve chat completions with an OpenAI-compatible API.")
print("token_count =", tok["count"])
print(json.dumps(tok["raw"], ensure_ascii=False, indent=2)[:600])


usage from response = {'prompt_tokens': 24, 'total_tokens': 35, 'completion_tokens': 11, 'prompt_tokens_details': None}
token_count = 15
{
  "count": 15,
  "max_model_len": 40960,
  "tokens": [
    85,
    4086,
    44,
    646,
    8683,
    6236,
    3459,
    908,
    448,
    458,
    5264,
    15469,
    80215,
    5333,
    13
  ],
  "token_strs": null
}


In [43]:
# 7) /metrics fetch + parsing (for throughput estimation in Experiment 3)
_metric_line_re = re.compile(
    r"^([a-zA-Z_:][a-zA-Z0-9_:]*)\{[^}]*\}\s+([0-9eE+\-.]+)$|^([a-zA-Z_:][a-zA-Z0-9_:]*)\s+([0-9eE+\-.]+)$"
)

def fetch_metrics_text():
    r = requests.get(_url("/metrics"), timeout=20)
    r.raise_for_status()
    return r.text

def parse_metrics(metrics_text):
    out = {}
    for line in metrics_text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        m = _metric_line_re.match(line)
        if not m:
            continue
        name = m.group(1) or m.group(3)
        val = m.group(2) or m.group(4)
        try:
            out[name] = float(val)
        except ValueError:
            pass
    return out

def metrics_snapshot():
    text = fetch_metrics_text()
    return {"t": time.perf_counter(), "metrics": parse_metrics(text), "raw_text": text}

snap = metrics_snapshot()
for k in ["vllm:prompt_tokens_total", "vllm:generation_tokens_total", "vllm:num_requests_running"]:
    if k in snap["metrics"]:
        print(k, "=", snap["metrics"][k])


vllm:prompt_tokens_total = 10020.0
vllm:generation_tokens_total = 26437.0
vllm:num_requests_running = 0.0


In [44]:
# 8) Small utilities for repeated runs, scoring, and simple benchmarking
def percentile(values, p):
    if not values:
        return float("nan")
    xs = sorted(values)
    if len(xs) == 1:
        return xs[0]
    rank = (len(xs) - 1) * (p / 100)
    lo = int(math.floor(rank))
    hi = int(math.ceil(rank))
    if lo == hi:
        return xs[lo]
    frac = rank - lo
    return xs[lo] * (1 - frac) + xs[hi] * frac

def run_repeated(prompt, repeats=3, **chat_kwargs):
    rows = [chat_once(prompt, **chat_kwargs) for _ in range(repeats)]
    texts = [r["text"].strip() for r in rows]
    return {
        "rows": rows,
        "texts": texts,
        "unique_count": len(set(texts)),
        "avg_len_chars": mean([len(t) for t in texts]) if texts else 0.0,
        "avg_latency_s": mean([r["latency_s"] for r in rows]) if rows else float("nan"),
    }

def run_batch_concurrent(prompt, *, n_requests=8, concurrency=4, **chat_kwargs):
    def _one(_i):
        return chat_once(prompt, **chat_kwargs)

    t0 = time.perf_counter()
    out = []
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futures = [ex.submit(_one, i) for i in range(n_requests)]
        for fut in as_completed(futures):
            out.append(fut.result())
    wall_s = time.perf_counter() - t0

    lats = [x["latency_s"] for x in out]
    usage_sum = {
        "prompt_tokens": sum((x["usage"].get("prompt_tokens") or 0) for x in out),
        "completion_tokens": sum((x["usage"].get("completion_tokens") or 0) for x in out),
        "total_tokens": sum((x["usage"].get("total_tokens") or 0) for x in out),
    }
    return {
        "rows": out,
        "wall_s": wall_s,
        "mean_s": mean(lats) if lats else None,
        "p50_s": percentile(lats, 50),
        "p95_s": percentile(lats, 95),
        "usage_sum": usage_sum,
    }

def estimate_gen_tokens_per_sec(metrics_before, metrics_after, wall_s, metric_name="vllm:generation_tokens_total"):
    b = metrics_before["metrics"].get(metric_name)
    a = metrics_after["metrics"].get(metric_name)
    if b is None or a is None or wall_s <= 0:
        return None
    return (a - b) / wall_s

_ = chat_once("warmup", max_tokens=MAX_TOKENS, enable_thinking=False)
m0 = metrics_snapshot()
demo_batch = run_batch_concurrent(
    "Give one short study tip in <= 12 words. One sentence only.",
    n_requests=6,
    concurrency=2,
    temperature=0.7,
    top_p=1.0,
    max_tokens=MAX_TOKENS,
    enable_thinking=False,
)
m1 = metrics_snapshot()
tps = estimate_gen_tokens_per_sec(m0, m1, demo_batch["wall_s"])
print({
    "wall_s": round(demo_batch["wall_s"], 3),
    "p50_s": round(demo_batch["p50_s"], 3),
    "p95_s": round(demo_batch["p95_s"], 3),
    "usage_sum": demo_batch["usage_sum"],
    "gen_tokens_per_s_est": None if tps is None else round(tps, 2),
})


{'wall_s': 0.364, 'p50_s': 0.115, 'p95_s': 0.129, 'usage_sum': {'prompt_tokens': 168, 'completion_tokens': 78, 'total_tokens': 246}, 'gen_tokens_per_s_est': 214.44}


## Part 2. Lab Tasks (single service, one model, three reports)
All three experiments below run on the **same server** (`http://localhost:8000`) and the **same model**.

You may use the provided helper functions directly, or write your own code.


### Experiment 1 — Generation Quality + Thinking ON/OFF (GSM8K-style mini set)
**Core goal:** Compare correctness and behavior with **thinking ON vs OFF** on a few GSM8K-style math word problems with known answers.

#### Small hints
- Keep decoding settings fixed (recommended: `temperature=0.0`, `top_p=1.0`)
- Use the same prompt format for all questions
- Ask for a strict final-answer format, e.g. `#### <number>`
- Compare:
  - exact-answer accuracy
  - average latency
  - example outputs / failure cases

#### Why this experiment matters
This directly tests whether enabling/disabling thinking changes **reasoning correctness** and **inference cost** on the same model/service.


In [46]:
gsm8k_mini = [
    {
        "id": "q1_algebra_system",
        "question": (
            "Real numbers x and y satisfy x + y = 14 and x^2 + y^2 = 130. "
            "Find |x - y|."
        ),
        "answer": 8,
    },
    {
        "id": "q2_arithmetic_sequence",
        "question": (
            "An arithmetic sequence has first term 5 and common difference 4. "
            "The sum of the first n terms is 434. Find n."
        ),
        "answer": 14,
    },
    {
        "id": "q3_geometry_chord",
        "question": (
            "A circle has radius 13. A chord is 5 units away from the center "
            "(measured along the perpendicular from the center to the chord). "
            "What is the length of the chord?"
        ),
        "answer": 24,
    },
    {
        "id": "q4_combinatorics_permutation",
        "question": (
            "How many distinct permutations are there of the word MISSISSIPPI?"
        ),
        "answer": 34650,
    },
    {
        "id": "q5_cards_exact_aces",
        "question": (
            "How many 5-card hands from a standard 52-card deck contain exactly 2 aces?"
        ),
        "answer": 103776,
    },
    {
        "id": "q6_calculus_definite_integral",
        "question": (
            "Compute the definite integral: integral from 0 to 2 of (3x^3 - 2x + 1) dx."
        ),
        "answer": 10,
    },
    {
        "id": "q7_linear_algebra_determinant",
        "question": (
            "Let A = [[2, 1], [1, 2]]. Find det(A^5)."
        ),
        "answer": 243,
    },
    {
        "id": "q8_number_theory_mod",
        "question": (
            "What is the remainder when 7^100 is divided by 13?"
        ),
        "answer": 9,
    },
    {
        "id": "q9_mixture_concentration",
        "question": (
            "A lab has 30 liters of a 20% salt solution. How many liters of pure water "
            "should be added to make the solution 12% salt?"
        ),
        "answer": 20,
    },
    {
        "id": "q10_geometry_word_problem",
        "question": (
            "A right triangle has legs that differ by 6. Its area is 216. "
            "What is the perimeter of the triangle?"
        ),
        "answer": 72,
    },
]

def build_math_prompt(question):
    return (
        "Solve the math word problem. You may think step by step, but the final line MUST be exactly in the format: #### <integer>\n\n"
        f"Question: {question}"
    )

_final_answer_re = re.compile(r"####\s*([-+]?\d+)")
_any_int_re = re.compile(r"[-+]?\d+")

def extract_final_int(text):
    m = _final_answer_re.search(text)
    if m:
        return int(m.group(1))
    ints = _any_int_re.findall(text)
    if ints:
        return int(ints[-1])
    return None

def eval_thinking_mode(dataset, enable_thinking, temperature=0.0, top_p=1.0):
    rows = []
    for item in dataset:
        prompt = build_math_prompt(item["question"])
        out = chat_once(
            prompt,
            temperature=temperature,
            top_p=top_p,
            max_tokens=MAX_TOKENS,
            enable_thinking=enable_thinking,
            timeout=240,
        )
        pred = extract_final_int(out["text"])
        rows.append({
            "id": item["id"],
            "answer_gt": item["answer"],
            "answer_pred": pred,
            "correct": int(pred == item["answer"]),
            "latency_s": out["latency_s"],
            "text": out["text"],
            "usage": out["usage"],
        })
    acc = sum(r["correct"] for r in rows) / len(rows) if rows else 0.0
    avg_latency = mean([r["latency_s"] for r in rows]) if rows else float("nan")
    return {"rows": rows, "accuracy": acc, "avg_latency_s": avg_latency}

# Run ON vs OFF (can take time because max_tokens is fixed at 4096)
res_on = eval_thinking_mode(gsm8k_mini, enable_thinking=True)
res_off = eval_thinking_mode(gsm8k_mini, enable_thinking=False)

print({
    "thinking_on_accuracy": round(res_on["accuracy"], 3),
    "thinking_on_avg_latency_s": round(res_on["avg_latency_s"], 3),
    "thinking_off_accuracy": round(res_off["accuracy"], 3),
    "thinking_off_avg_latency_s": round(res_off["avg_latency_s"], 3),
})

for r_on, r_off in zip(res_on["rows"], res_off["rows"]):
    print("\n", r_on["id"], "GT =", r_on["answer_gt"])
    print("  ON : pred =", r_on["answer_pred"], "correct =", r_on["correct"], "lat =", round(r_on["latency_s"], 2))
    print("  OFF: pred =", r_off["answer_pred"], "correct =", r_off["correct"], "lat =", round(r_off["latency_s"], 2))


{'thinking_on_accuracy': 1.0, 'thinking_on_avg_latency_s': 2.348, 'thinking_off_accuracy': 0.8, 'thinking_off_avg_latency_s': 0.507}

 q1_algebra_system GT = 8
  ON : pred = 8 correct = 1 lat = 2.3
  OFF: pred = 8 correct = 1 lat = 0.62

 q2_arithmetic_sequence GT = 14
  ON : pred = 14 correct = 1 lat = 2.15
  OFF: pred = 14 correct = 1 lat = 0.69

 q3_geometry_chord GT = 24
  ON : pred = 24 correct = 1 lat = 0.77
  OFF: pred = 24 correct = 1 lat = 0.33

 q4_combinatorics_permutation GT = 34650
  ON : pred = 34650 correct = 1 lat = 2.69
  OFF: pred = 34650 correct = 1 lat = 0.43

 q5_cards_exact_aces GT = 103776
  ON : pred = 103776 correct = 1 lat = 6.49
  OFF: pred = 102 correct = 0 lat = 0.38

 q6_calculus_definite_integral GT = 10
  ON : pred = 10 correct = 1 lat = 2.21
  OFF: pred = 14 correct = 0 lat = 0.32

 q7_linear_algebra_determinant GT = 243
  ON : pred = 243 correct = 1 lat = 2.24
  OFF: pred = 243 correct = 1 lat = 0.48

 q8_number_theory_mod GT = 9
  ON : pred = 9 correc

#### Experiment 1 notes / pitfalls
- If outputs are long, keep `max_tokens=4096` but tighten the prompt format requirement.
- If the model does not follow `#### <integer>` exactly, your extractor can fall back to the last integer (already implemented above).
- If timing varies a lot, rerun and compare averages instead of single examples.


### Experiment 2 — Decoding Parameter Effects (stability vs diversity)
**Core goal:** Study how decoding parameters change output behavior, and recommend settings for a stable task and a creative task.

#### Small hints
- Keep `max_tokens=4096` fixed (lab rule)
- Control output length with prompt wording (e.g., “one sentence only”, “answer only”)
- Repeat each setting 3 times
- Use simple metrics:
  - unique count across 3 outputs (1/2/3)
  - average length
  - a short human observation


In [47]:
# Experiment 2 starter: two prompts + parameter grid
stable_prompt = "Answer only (no explanation): What is 84 + 19?"
creative_prompt = "Write one short slogan (<= 12 words) about fast inference. One sentence only."

param_grid = [
    {"temperature": 0.0, "top_p": 1.0, "top_k": 20, "max_tokens": MAX_TOKENS, "enable_thinking": False},
    {"temperature": 0.2, "top_p": 1.0, "top_k": 20, "max_tokens": MAX_TOKENS, "enable_thinking": False},
    {"temperature": 0.7, "top_p": 1.0, "top_k": 20, "max_tokens": MAX_TOKENS, "enable_thinking": False},
    {"temperature": 1.2, "top_p": 1.0, "top_k": 20, "max_tokens": MAX_TOKENS, "enable_thinking": False},
    {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "max_tokens": MAX_TOKENS, "enable_thinking": False},
]

rows_exp2 = []
for cfg in param_grid:
    s = run_repeated(stable_prompt, repeats=3, **cfg)
    c = run_repeated(creative_prompt, repeats=3, **cfg)
    rows_exp2.append({
        "cfg": cfg,
        "stable_unique_count": s["unique_count"],
        "stable_avg_len_chars": round(s["avg_len_chars"], 1),
        "stable_avg_latency_s": round(s["avg_latency_s"], 3),
        "creative_unique_count": c["unique_count"],
        "creative_avg_len_chars": round(c["avg_len_chars"], 1),
        "creative_avg_latency_s": round(c["avg_latency_s"], 3),
        "stable_sample": s["texts"][0][:120],
        "creative_sample": c["texts"][0][:120],
    })

for row in rows_exp2:
    print("\nCFG:", row["cfg"])
    print("  stable : unique =", row["stable_unique_count"], "avg_len =", row["stable_avg_len_chars"], "avg_lat =", row["stable_avg_latency_s"])
    print("  creative: unique =", row["creative_unique_count"], "avg_len =", row["creative_avg_len_chars"], "avg_lat =", row["creative_avg_latency_s"])
    print("  creative sample:", row["creative_sample"])



CFG: {'temperature': 0.0, 'top_p': 1.0, 'top_k': 20, 'max_tokens': 10240, 'enable_thinking': False}
  stable : unique = 1 avg_len = 13 avg_lat = 0.104
  creative: unique = 1 avg_len = 29 avg_lat = 0.095
  creative sample: Fast inference, fast results.

CFG: {'temperature': 0.2, 'top_p': 1.0, 'top_k': 20, 'max_tokens': 10240, 'enable_thinking': False}
  stable : unique = 1 avg_len = 13 avg_lat = 0.111
  creative: unique = 2 avg_len = 29.3 avg_lat = 0.102
  creative sample: Fast inference, fast results.

CFG: {'temperature': 0.7, 'top_p': 1.0, 'top_k': 20, 'max_tokens': 10240, 'enable_thinking': False}
  stable : unique = 1 avg_len = 13 avg_lat = 0.112
  creative: unique = 2 avg_len = 20 avg_lat = 0.095
  creative sample: Fast inference for the future.

CFG: {'temperature': 1.2, 'top_p': 1.0, 'top_k': 20, 'max_tokens': 10240, 'enable_thinking': False}
  stable : unique = 1 avg_len = 13 avg_lat = 0.113
  creative: unique = 3 avg_len = 25 avg_lat = 0.099
  creative sample: Fast Inference!

### Experiment 3 — Throughput / Latency / Capacity (single service)
**Core goal:** Measure how workload conditions affect latency and throughput on the same service, and choose a practical operating point.

#### Recommended design (two mini studies in one report)
1. **Concurrency sweep** (fixed prompt, fixed `max_tokens=4096`)
2. **Workload-type comparison** (same concurrency, compare easy short-answer vs reasoning-heavy / thinking ON)


In [48]:
# Experiment 3 starter A: concurrency sweep (single service, fixed prompt, fixed max_tokens=4096)
bench_prompt_short = "Answer only: What is 37 + 58?"

_ = chat_once("warmup", enable_thinking=False, max_tokens=MAX_TOKENS)

concurrency_levels = [1, 2, 4, 8]
n_requests = 8

rows_concurrency = []
for c in concurrency_levels:
    m0 = metrics_snapshot()
    res = run_batch_concurrent(
        bench_prompt_short,
        n_requests=n_requests,
        concurrency=c,
        temperature=0.0,
        top_p=1.0,
        top_k=20,
        max_tokens=MAX_TOKENS,
        enable_thinking=False,
    )
    m1 = metrics_snapshot()
    tps = estimate_gen_tokens_per_sec(m0, m1, res["wall_s"], metric_name="vllm:generation_tokens_total")
    rows_concurrency.append({
        "concurrency": c,
        "requests": n_requests,
        "p50_s": round(res["p50_s"], 3),
        "p95_s": round(res["p95_s"], 3),
        "wall_s": round(res["wall_s"], 3),
        "tokens_per_s_est": None if tps is None else round(tps, 2),
        "completion_tokens_sum": res["usage_sum"]["completion_tokens"],
    })

rows_concurrency


[{'concurrency': 1,
  'requests': 8,
  'p50_s': 0.103,
  'p95_s': 0.108,
  'wall_s': 0.835,
  'tokens_per_s_est': 105.34,
  'completion_tokens_sum': 88},
 {'concurrency': 2,
  'requests': 8,
  'p50_s': 0.103,
  'p95_s': 0.111,
  'wall_s': 0.42,
  'tokens_per_s_est': 209.77,
  'completion_tokens_sum': 88},
 {'concurrency': 4,
  'requests': 8,
  'p50_s': 0.105,
  'p95_s': 0.109,
  'wall_s': 0.216,
  'tokens_per_s_est': 408.18,
  'completion_tokens_sum': 88},
 {'concurrency': 8,
  'requests': 8,
  'p50_s': 0.117,
  'p95_s': 0.119,
  'wall_s': 0.124,
  'tokens_per_s_est': 712.06,
  'completion_tokens_sum': 88}]

In [49]:
# Experiment 3 starter B: workload-type comparison (same server, same max_tokens=4096)
workload_A_prompt = "Answer only: What is 146 - 59?"
workload_B_prompt = (
    "Solve the problem and output ONLY the final answer in the format '#### <number>'.\n"
    "Question: A shop had 72 apples, sold 19, then bought 34 more. How many apples does it have now?"
)

fixed_concurrency = 4
n_requests_w = 8

def run_profile(prompt, *, enable_thinking):
    m0 = metrics_snapshot()
    res = run_batch_concurrent(
        prompt,
        n_requests=n_requests_w,
        concurrency=fixed_concurrency,
        temperature=0.0,
        top_p=1.0,
        top_k=20,
        max_tokens=MAX_TOKENS,
        enable_thinking=enable_thinking,
        timeout=240,
    )
    m1 = metrics_snapshot()
    tps = estimate_gen_tokens_per_sec(m0, m1, res["wall_s"], metric_name="vllm:generation_tokens_total")
    return {
        "p50_s": round(res["p50_s"], 3),
        "p95_s": round(res["p95_s"], 3),
        "wall_s": round(res["wall_s"], 3),
        "tokens_per_s_est": None if tps is None else round(tps, 2),
        "usage_sum": res["usage_sum"],
    }

profile_A = run_profile(workload_A_prompt, enable_thinking=False)
profile_B = run_profile(workload_B_prompt, enable_thinking=True)

print("Profile A (short answer, thinking OFF):", profile_A)
print("Profile B (reasoning prompt, thinking ON):", profile_B)


Profile A (short answer, thinking OFF): {'p50_s': 0.137, 'p95_s': 0.139, 'wall_s': 0.276, 'tokens_per_s_est': 347.84, 'usage_sum': {'prompt_tokens': 208, 'completion_tokens': 96, 'total_tokens': 304}}
Profile B (reasoning prompt, thinking ON): {'p50_s': 1.019, 'p95_s': 1.086, 'wall_s': 2.11, 'tokens_per_s_est': 2162.09, 'usage_sum': {'prompt_tokens': 448, 'completion_tokens': 4563, 'total_tokens': 5011}}


## Report Templates (bottom of notebook, copy-paste into 3 Markdown files)
Create three files:
- `report_exp1_generation_thinking.md`
- `report_exp2_decoding.md`
- `report_exp3_throughput.md`


### Template 1 — `report_exp1_generation_thinking.md`
```markdown
# Experiment 1 Report: Generation Quality + Thinking ON/OFF (GSM8K-style mini set)

## 1. Objective
- [...]

## 2. Setup
- Server: `http://localhost:8000`
- Model ID: `[...]`
- Fixed decoding settings: temperature=[...], top_p=[...], top_k=[...], max_tokens=4096
- Thinking mode comparison:
  - ON: `chat_template_kwargs={"enable_thinking": true}`
  - OFF: `chat_template_kwargs={"enable_thinking": false}`
- Dataset size: [...] questions
- Final-answer extraction rule (e.g., `#### <integer>` + fallback): [...]

## 3. Results
| Question ID | Ground Truth | Pred (Thinking ON) | Correct ON | Pred (Thinking OFF) | Correct OFF | Latency ON (s) | Latency OFF (s) | Notes |
|---|---:|---:|---:|---:|---:|---:|---:|---|
| ... | ... | ... | ... | ... | ... | ... | ... | ... |

## 4. Summary Metrics
- Accuracy (Thinking ON): [...]
- Accuracy (Thinking OFF): [...]
- Avg latency (Thinking ON): [...]
- Avg latency (Thinking OFF): [...]

## 5. Analysis (4–8 sentences)
- Did thinking mode improve accuracy on your mini set?
- What was the latency trade-off?
- Which problem types benefited most / failed most?
- What would you choose for correctness-sensitive tasks?
```


### Template 2 — `report_exp2_decoding.md`
```markdown
# Experiment 2 Report: Decoding Parameter Effects

## 1. Objective
- [...]

## 2. Setup
- Stable prompt: [...]
- Creative prompt: [...]
- Fixed rule: `max_tokens=4096`
- Parameter configurations tested (4–8):
  - [...]
- Repeats per configuration: 3

## 3. Results
| Scenario | Params (temperature/top_p/top_k/...) | Repeats | Unique Count (1/2/3) | Avg Length | Avg Latency (s) | Conclusion (1 sentence) |
|---|---|---:|---:|---:|---:|---|
| Stable | ... | 3 | ... | ... | ... | ... |
| Stable | ... | 3 | ... | ... | ... | ... |
| Creative | ... | 3 | ... | ... | ... | ... |
| Creative | ... | 3 | ... | ... | ... | ... |

## 4. Recommended Settings + Analysis (4–8 sentences)
- BEST_STABLE: [...] (reason: [...])
- BEST_CREATIVE: [...] (reason: [...])
- Which parameter(s) affected diversity the most?
- What trade-off did you observe (stability, diversity, latency, format compliance)?
```


### Template 3 — `report_exp3_throughput.md`
```markdown
# Experiment 3 Report: Throughput / Latency / Capacity (Single Service)

## 1. Objective
- [...]

## 2. Setup
- Server: `http://localhost:8000`
- Fixed rule: `max_tokens=4096`
- Warmup: [Yes/No]
- Metric used for throughput estimate (from `/metrics`): `[actual metric name]`
- Throughput formula: `(metric_after - metric_before) / wall_time`

## 3. Study A: Concurrency Sweep (fixed prompt)
- Benchmark prompt: [...]
- Requests per level: [...]

| Concurrency | Requests | p50 (s) | p95 (s) | Wall Time (s) | Estimated tokens/s | Notes |
|---:|---:|---:|---:|---:|---:|---|
| ... | ... | ... | ... | ... | ... | ... |
| ... | ... | ... | ... | ... | ... | ... |

## 4. Study B: Workload-Type Comparison (same concurrency)
- Fixed concurrency: [...]
- Workload A (short-answer / thinking OFF): [...]
- Workload B (reasoning-heavy or thinking ON): [...]

| Workload | Thinking Mode | Requests | p50 (s) | p95 (s) | Wall Time (s) | Estimated tokens/s | Notes |
|---|---|---:|---:|---:|---:|---:|---|
| A | OFF | ... | ... | ... | ... | ... | ... |
| B | ON  | ... | ... | ... | ... | ... | ... |

## 5. Analysis + Recommendation (4–8 sentences)
- How did latency change with concurrency?
- Where is the knee point (if any)?
- How different are short-answer vs reasoning-heavy workloads on the same server?
- What operating point would you recommend for a classroom/demo server?
```


In [ ]:
#1.延迟通常随并发量的增加而增加。低并发的时候，因为服务器资源充足，延迟较低且增长缓慢；高并发时，延迟急剧上升，甚至出现超时或拒绝服务的情况，系统达到瓶颈。
#2.拐点的位置无法预测，通常发生在资源利用率达到百分之八九十的时候。因此，我们通常通过绘制延迟随并发量改变而改变的曲线来寻找拐点。
#3.短答案工作负载通常只是简单的查询或响应，计算量小、耗时短，对资源占用低；推理密集型工作负载涉及复杂计算，计算量大、耗时长，资源占用高，容易成为瓶颈。
#当它们共存时，在同一服务器上，推理密集型任务会拖慢整体响应速度，导致短答案任务也需要排队等待，整体延迟被拉高。
#4.相比拐点差距一部分的一些点，例如资源利用率百分之六七十的时候，这样可以多预留一点余量来应对此类情景可能出现的事件